# Intent Classification Pipeline

This notebook prepares user messages, merges batch classification results, retries failed cases, and exports the final dataset.

In [1]:
import os
import json
from tqdm import tqdm
import pandas as pd

searches = json.load(open("../data/searches.json"))
sha_to_repo = {entry["sha"]: entry["repository"]["full_name"] for entry in searches}

rows = []
for filename in tqdm(os.listdir("../data/parsed_chats_simple")):
    if filename.endswith(".json"):
        with open(
            os.path.join("../data/parsed_chats_simple", filename), "r", encoding="utf-8"
        ) as f:
            chat_data = json.load(f)
        sha = filename.replace(".md.json", "")
        rows.append(
            {
                "sha": sha,
                "title": chat_data.get("title"),
                "timestamp": chat_data.get("timestamp"),
                "repository_full_name": sha_to_repo.get(sha),
            }
        )
df_chats = pd.DataFrame(rows)

100%|██████████| 11655/11655 [00:16<00:00, 725.59it/s]


In [2]:
def extract_messages():
    rows = []
    folder = "../data/parsed_chats_simple"
    for filename in tqdm(os.listdir(folder)):
        if filename.endswith(".json"):
            chat_data = json.load(open(os.path.join(folder, filename)))
            messages = chat_data.get("messages", [])
            for i, msg in enumerate(messages):
                rows.append(
                    {
                        "sha": filename.replace(".md.json", ""),
                        "index": i,
                        "role": msg.get("role"),
                        "content": msg.get("content"),
                    }
                )
    return pd.DataFrame(rows)


df_messages = extract_messages()
df_messages = df_messages.merge(df_chats, on="sha", how="left")
df_messages = df_messages.sort_values(
    by=["repository_full_name", "timestamp", "sha", "index"]
).reset_index(drop=True)
assert (
    df_messages.groupby("sha")["repository_full_name"].nunique().max() == 1
), "Error: Some sha have multiple repository_full_name"
assert (
    df_messages.groupby("sha")["timestamp"].nunique().max() == 1
), "Error: Some sha have multiple timestamp"

100%|██████████| 11655/11655 [00:15<00:00, 774.71it/s]


In [3]:
def truncate(text: str, limit: int = 1000) -> str:
    if not isinstance(text, str):
        return ""

    normalized = text.replace("\n", " ")
    if len(normalized) <= limit:
        return normalized
    return normalized[: limit * 3 // 5] + "..." + normalized[-limit * 2 // 5 :]


def format_message_line(role, content) -> str:
    safe_role = role if isinstance(role, str) else "Unknown"
    return f"[prev_{safe_role.lower()}]: {truncate(content, limit=1000)}"


df_messages["context"] = ""
num_sessions = df_messages["sha"].nunique()

for _, chat_df in tqdm(
    df_messages.groupby("sha", sort=False),
    total=num_sessions,
    desc="Building message context",
):
    chat_df = chat_df.sort_values("index")

    session_contexts = []
    last_two_lines = []

    for row in chat_df.itertuples(index=True):
        session_contexts.append("\n".join(last_two_lines))

        current_line = format_message_line(row.role, row.content)
        last_two_lines.append(current_line)
        if len(last_two_lines) > 2:
            last_two_lines.pop(0)

    df_messages.loc[chat_df.index, "context"] = session_contexts

Building message context: 100%|██████████| 11655/11655 [00:24<00:00, 476.05it/s]


In [4]:
df_user_messages = df_messages[df_messages["role"] == "User"].reset_index(drop=True)
df_user_messages.drop(columns=["role"], inplace=True)
df_user_messages["truncated_content"] = df_user_messages["content"].apply(
    lambda x: truncate(x, limit=10000)
)
os.makedirs("../data/classifications", exist_ok=True)
df_user_messages.to_csv(
    "../data/classifications/all_user_messages_cleaned.csv", index=False
)

**Run `classification_batch.py prepare` then `classification_batch_ops.sh submit` (and later `download`) before executing the following cells.**

## Batch Classification Stage

After generating `all_user_messages_cleaned.csv`, run the external batch scripts, then continue with the cells below to merge and evaluate results.

In [5]:
from pathlib import Path

parsed_dir = Path("../data/classifications/parsed")
all_files = sorted(parsed_dir.glob("classifications_*.jsonl"))
if not all_files:
    raise FileNotFoundError(f"No parsed classification files found in: {parsed_dir}")

all_lines = []
for _f in all_files:
    lines = _f.read_text(encoding="utf-8").strip().split("\n")
    all_lines.extend(lines)

output_path = "../data/classifications/all_classifications.jsonl"
with open(output_path, "w", encoding="utf-8") as _fout:
    _fout.write("\n".join(all_lines) + "\n")

print(f"Merged {len(all_files)} file(s) → {len(all_lines)} lines → {output_path}")

Merged 763 file(s) → 76231 lines → ../data/classifications/all_classifications.jsonl


In [6]:
df_classifications = pd.read_json(
    "../data/classifications/all_classifications.jsonl", lines=True
)
df_classifications = df_classifications.sort_values("index").reset_index(drop=True)
df_classifications["status"].value_counts()

failed_indexes = df_classifications[df_classifications["status"] != "success"][
    "index"
].tolist()

output_dir = "../data/classifications/fail"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/failed_indexes.json"
with open(output_path, "w") as f:
    json.dump(failed_indexes, f, indent=2)

print(f"Saved {len(failed_indexes)} failed indexes to {output_path}")

Saved 93 failed indexes to ../data/classifications/fail/failed_indexes.json


**Run `classification_failed.py` before executing the following cells.**

## Retry and Consolidation

This section replaces failed rows with retry outputs and rebuilds a single cleaned classification table.

In [7]:
df_failed_retry = pd.read_json(
    f"{output_dir}/classifications_failed_retry.jsonl", lines=True
)
df_failed_retry.drop(columns=["attempt"], inplace=True)

replace_cols = [
    c
    for c in df_failed_retry.columns
    if c in df_classifications.columns and c != "index"
]

df_classifications = df_classifications.set_index("index")
df_failed_retry_indexed = df_failed_retry.set_index("index")

replace_index = df_failed_retry_indexed.index.intersection(df_classifications.index)
df_classifications.loc[replace_index, replace_cols] = df_failed_retry_indexed.loc[
    replace_index, replace_cols
]

df_classifications = (
    df_classifications.reset_index().sort_values("index").reset_index(drop=True)
)
df_classifications.drop(columns=["index"], inplace=True)
print(f"Replaced {len(replace_index)} rows in df_classifications by index")
print(f"Updated status value counts:\n{df_classifications['status'].value_counts()}")

Replaced 93 rows in df_classifications by index
Updated status value counts:
status
success    76231
Name: count, dtype: int64


In [8]:
df_user_messages_with_classifications = pd.concat(
    [df_user_messages, df_classifications], axis=1
)

df_user_messages_with_classifications["index_in_chat"] = (
    df_user_messages_with_classifications.groupby("sha").cumcount()
)

df_cleaned = df_user_messages_with_classifications[
    [
        "repository_full_name",
        "sha",
        "timestamp",
        "title",
        "index_in_chat",
        "truncated_content",
        "labels",
        "content",
    ]
]
df_cleaned.to_csv(
    "../data/classifications/classifications_with_messages.csv", index=False
)

## Cost Summary

Estimate total API cost from prompt, cached, and completion token usage.

In [9]:
total_prompt_tokens = df_classifications["prompt_tokens"].sum()
total_completion_tokens = df_classifications["completion_tokens"].sum()
total_cached_tokens = df_classifications["cached_tokens"].sum()
# Model: gpt-5-mini
price_input_per_1m = 0.25
price_cached_input_per_1m = 0.025
price_output_per_1m = 2.00
total_cost = (
    (total_prompt_tokens - total_cached_tokens) * price_input_per_1m
    + total_completion_tokens * price_output_per_1m
    + total_cached_tokens * price_cached_input_per_1m
) / 1_000_000
batch_estimated_cost = total_cost * 0.5
print(f"Total prompt tokens: {total_prompt_tokens}")
print(f"Total completion tokens: {total_completion_tokens}")
print(f"Total cached tokens: {total_cached_tokens}")
print(f"Total cost (standard rates): ${total_cost:.4f}")
print(f"Estimated cost via Batch API (50% off): ${batch_estimated_cost:.4f}")

Total prompt tokens: 332672408
Total completion tokens: 34935118
Total cached tokens: 292700928
Total cost (standard rates): $87.1806
Estimated cost via Batch API (50% off): $43.5903
